# Lagrangian advection on the FESOM double gyre (daily output)

This is the FESOM counterpart of [`cmems_global/02_run_parcels.ipynb`](../cmems_global/02_run_parcels.ipynb):
seed particles in a velocity field and advect them with Parcels v4. The
difference is the grid — CMEMS is a structured (sgrid) global product, while
FESOM2 writes its horizontal velocities on the **centres of triangles** of an
unstructured mesh, so we go through the `uxarray` / UGRID path instead of
`copernicusmarine_to_sgrid`.

The double-gyre output already lives on DKRZ disk
(`/work/bk1450/b383184/ELPHE_hackathon/Double_gyre/FESOM_v2.7/`), so there is no
download step (no `01_retrieve_data` equivalent) — we read it in place.

We reuse the **cell-view linear reconstruction** developed in
[`../fesom_reconstruction/fesom_velocity_reconstruction.ipynb`](../fesom_reconstruction/fesom_velocity_reconstruction.ipynb):
the face-registered FESOM velocities are turned into a linear field per triangle
(least-squares fit to edge neighbours) instead of Parcels' piecewise-constant
default. Set `USE_RECON = False` below to fall back to the built-in interpolator.

> **Environment** — run this with the `fesom-reconstruction` pixi environment
> (`../fesom_reconstruction/pixi.toml`): Parcels v4 + uxarray. From that folder,
> `pixi run kernel` registers the Jupyter kernel.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import numpy as np
import parcels
import uxarray as ux
import xarray as xr
from uxarray.constants import INT_FILL_VALUE

print("parcels", parcels.__version__)
print("uxarray", ux.__version__)

## Parameters

`fesom.mesh.diag.nc` provides the mesh topology (the UGRID grid); the `u`/`v`/`w`
files carry the daily velocity snapshots for 1950. FESOM stores `u`, `v` on
**elements** (triangle centres → Parcels faces) and `w` on **nodes**; that is
exactly what the reconstruction kernel expects. The `unod`/`vnod` files are the
node-interpolated horizontals — we do *not* use them here.

`MESH` must match how the double-gyre coordinates are stored. Idealised FESOM
double-gyre setups are usually on a sphere in degrees (`"spherical"`); if the
inspection cell below shows coordinates that are not longitudes/latitudes (e.g.
metres or a small Cartesian box), switch to `"flat"`.

In [ ]:
DATA_ROOT = Path("/work/bk1450/b383184/ELPHE_hackathon/Double_gyre/FESOM_v2.7")
FREQ = "1d"  # daily output

GRID_PATH = str(DATA_ROOT / "fesom.mesh.diag.nc")
DATA_PATHS = [str(DATA_ROOT / FREQ / f"{v}.fesom.1950.nc") for v in ("u", "v", "w")]

MESH = "spherical"  # set to "flat" if coords are Cartesian (see inspection cell)
USE_RECON = True  # cell-view linear reconstruction vs. piecewise-constant default
N_SURFACE_LEVELS = 3  # keep only the top few layers in memory (surface advection)

print("grid:", GRID_PATH)
for p in DATA_PATHS:
    print("data:", p)

## Reconstruction kernel

Lifted verbatim from the reconstruction notebook (and `icon/profile_icon.py`):
purely geometric least-squares gradient coefficients computed once and cached on
the `uxgrid`, plus the interpolator that reads them back. See that notebook for
the derivation.

In [ ]:
DEG2M = 1852.0 * 60.0  # metres per degree (matches Ux_Velocity unit conversion)


def local_offsets(mesh, lon_c, lat_c, lon, lat):
    # local planar offset (dx, dy) of (lon, lat) from centroid (lon_c, lat_c)
    if mesh == "spherical":
        dlon = ((np.asarray(lon) - lon_c + 180.0) % 360.0) - 180.0
        dx = dlon * np.cos(np.deg2rad(lat_c)) * DEG2M
        dy = (np.asarray(lat) - lat_c) * DEG2M
    else:  # flat mesh: coordinates are already a planar (x, y)
        dx = np.asarray(lon) - lon_c
        dy = np.asarray(lat) - lat_c
    return dx, dy


def build_reconstruction_terms(uxgrid, mesh):
    """Cache cell-view least-squares gradient coefficients on uxgrid._ds."""
    nbr = np.asarray(uxgrid.face_face_connectivity.values)  # (F, K)
    flon = np.asarray(uxgrid.face_lon.values)
    flat = np.asarray(uxgrid.face_lat.values)

    valid = nbr != INT_FILL_VALUE
    nbr_safe = np.where(valid, nbr, 0)

    dx, dy = local_offsets(
        mesh, flon[:, None], flat[:, None], flon[nbr_safe], flat[nbr_safe]
    )
    dx = np.where(valid, dx, 0.0)
    dy = np.where(valid, dy, 0.0)

    X2 = np.sum(dx * dx, axis=1)
    Y2 = np.sum(dy * dy, axis=1)
    XY = np.sum(dx * dy, axis=1)
    d = X2 * Y2 - XY * XY

    ok = d != 0.0  # < 2 independent neighbours -> zero-gradient fallback
    inv_d = np.where(ok, 1.0 / np.where(ok, d, 1.0), 0.0)

    gx = (dx * Y2[:, None] - dy * XY[:, None]) * inv_d[:, None]
    gy = (dy * X2[:, None] - dx * XY[:, None]) * inv_d[:, None]
    gx = np.where(valid, gx, 0.0)
    gy = np.where(valid, gy, 0.0)

    dims = ("n_face", "n_max_face_faces")
    uxgrid._ds["gradient_coeff_x"] = xr.DataArray(gx, dims=dims)
    uxgrid._ds["gradient_coeff_y"] = xr.DataArray(gy, dims=dims)
    uxgrid._ds["recon_face_neighbors"] = xr.DataArray(nbr_safe, dims=dims)
    uxgrid._ds["recon_neighbor_mask"] = xr.DataArray(valid.astype(np.float64), dims=dims)
    return uxgrid


def LinearFaceRecon(particle_positions, grid_positions, field):
    ti = grid_positions["T"]["index"]
    zi = grid_positions["Z"]["index"]
    fi = grid_positions["FACE"]["index"]

    uxg = field.grid.uxgrid
    gx = uxg._ds["gradient_coeff_x"].values[fi]  # (M, K)
    gy = uxg._ds["gradient_coeff_y"].values[fi]
    nb = uxg._ds["recon_face_neighbors"].values[fi]
    mask = uxg._ds["recon_neighbor_mask"].values[fi]

    uc = field.data.values[ti, zi, fi]  # (M,)
    un = field.data.values[ti[:, None], zi[:, None], nb]  # (M, K)
    du = (un - uc[:, None]) * mask
    ax = np.sum(gx * du, axis=1)
    ay = np.sum(gy * du, axis=1)

    flon = uxg.face_lon.values[fi]
    flat = uxg.face_lat.values[fi]
    dx, dy = local_offsets(
        field.grid._mesh, flon, flat, particle_positions["lon"], particle_positions["lat"]
    )
    return uc + ax * dx + ay * dy

## Build the FieldSet

`ux.open_mfdataset(grid, data)` reads the mesh topology together with the daily
velocity files; `fesom_to_ugrid` renames the FESOM dimensions to the UGRID names
Parcels expects (`nz1 -> zc`, etc.). We keep only `U`, `V`, `W` and slice to the
top `N_SURFACE_LEVELS` layers to bound memory — the full 3-D `u`/`v` are ~3.8 GB
each.

In [ ]:
fds = ux.open_mfdataset(GRID_PATH, DATA_PATHS).rename_vars(
    {"u": "U", "v": "V", "w": "W"}
)
fds = parcels.convert.fesom_to_ugrid(fds)
fds

In [ ]:
# Inspect coordinates to confirm MESH ("spherical" vs "flat") and the domain.
print("vertical centers zc (m):", np.asarray(fds.zc.values)[:5], "...")
print("n time steps:", fds.sizes.get("time"))
g = fds.uxgrid
print(f"lon range: {float(g.node_lon.min()):.3f} .. {float(g.node_lon.max()):.3f}")
print(f"lat range: {float(g.node_lat.min()):.3f} .. {float(g.node_lat.max()):.3f}")
print(f"{g.n_face} faces (elements), {g.n_node} nodes")

In [ ]:
# Surface layers only -> much smaller in-memory footprint.
fds = fds.isel(zc=slice(0, N_SURFACE_LEVELS))
fesom = parcels.FieldSet.from_ugrid_conventions(fds, mesh=MESH)

if USE_RECON:
    build_reconstruction_terms(fesom.fields["U"].grid.uxgrid, mesh=MESH)
    fesom.fields["U"].interp_method = LinearFaceRecon
    fesom.fields["V"].interp_method = LinearFaceRecon
    n_nbr = fesom.fields["U"].grid.uxgrid._ds["recon_neighbor_mask"].values.sum(axis=1)
    print(f"reconstruction on: {(n_nbr == 3).sum()} interior (3-neighbour) triangles")
else:
    print("using built-in piecewise-constant face interpolator")

print(fesom)

## Seed particles

We lay a regular lon/lat grid of particles across the interior of the domain
(derived from the mesh extents, with a small margin to avoid boundary triangles)
and release them at the surface at the first available time. Increase
`N_LON x N_LAT` to scale up the experiment.

In [ ]:
N_LON, N_LAT = 60, 60  # 3600 particles; raise for a larger experiment

g = fesom.fields["U"].grid.uxgrid
lon_lo, lon_hi = float(g.node_lon.min()), float(g.node_lon.max())
lat_lo, lat_hi = float(g.node_lat.min()), float(g.node_lat.max())
mlon = 0.05 * (lon_hi - lon_lo)
mlat = 0.05 * (lat_hi - lat_lo)

lon_grid, lat_grid = np.meshgrid(
    np.linspace(lon_lo + mlon, lon_hi - mlon, N_LON),
    np.linspace(lat_lo + mlat, lat_hi - mlat, N_LAT),
)
z_surface = float(fds.zc.values[0])
t0 = fds.time.values[0]

pset = parcels.ParticleSet(
    fesom,
    pclass=parcels.Particle,
    lon=lon_grid.ravel(),
    lat=lat_grid.ravel(),
    z=np.full(lon_grid.size, z_surface),
    time=np.full(lon_grid.size, t0),
)
print(pset)

## Run the advection

Daily fields, so a daily output cadence over ~30 days with a short internal
time step. Adjust `runtime` up to the length of the 1950 record.

In [ ]:
output_file = parcels.ParticleFile(
    "fesom_double_gyre_trajectories.parquet",
    outputdt=np.timedelta64(1, "D"),
    mode="w",
)

pset.execute(
    [parcels.kernels.AdvectionRK4],
    runtime=np.timedelta64(30, "D"),
    dt=np.timedelta64(30, "m"),
    output_file=output_file,
)

In [ ]:
df = parcels.read_particlefile("fesom_double_gyre_trajectories.parquet")
df

## Plot the trajectories over the surface speed field

In [ ]:
triang = mtri.Triangulation(
    g.node_lon.values, g.node_lat.values, triangles=g.face_node_connectivity.values
)
speed = np.hypot(
    np.asarray(fds["U"].isel(time=0, zc=0)).squeeze(),
    np.asarray(fds["V"].isel(time=0, zc=0)).squeeze(),
)

fig, ax = plt.subplots(figsize=(11, 9))
ax.tripcolor(triang, facecolors=speed, shading="flat", cmap="Blues", alpha=0.7)

_df = df.to_pandas().sort_values("time")
sc = ax.scatter(
    _df["lon"], _df["lat"], c=_df["time"], s=1, alpha=0.5, cmap="viridis_r"
)
ax.set_xlabel("Longitude [deg E]" if MESH == "spherical" else "x")
ax.set_ylabel("Latitude [deg N]" if MESH == "spherical" else "y")
ax.set_title("FESOM double-gyre trajectories (color = time)")
fig.colorbar(sc, ax=ax, label="time")
plt.show()